In [1]:
!pip install spotipy

import spotipy

import pandas as pd
from spotipy.oauth2 import SpotifyOAuth

  Obtaining dependency information for redis>=3.5.3 from https://files.pythonhosted.org/packages/0b/34/a01250ac1fc9bf9161e07956d2d580413106ce02d5591470130a25c599e3/redis-5.0.1-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.3/250.3 kB 6.6 MB/s eta 0:00:00


In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("spotify_api_key")
secret_value_1 = user_secrets.get_secret("spotify_client_id")
secret_value_2 = user_secrets.get_secret("spotify_playlist_driveby")

In [3]:
import requests
import base64

CLIENT_ID=secret_value_1
CLIENT_SECRET=secret_value_0

client_credentials=f"{CLIENT_ID}:{CLIENT_SECRET}"
client_credentials_base64 = base64.b64encode(client_credentials.encode())

token_url = 'https://accounts.spotify.com/api/token'
headers = {
    'Authorization': f'Basic {client_credentials_base64.decode()}'
}
data = {
    'grant_type': 'client_credentials'
}
response = requests.post(token_url, data=data, headers=headers)

if response.status_code == 200:
    access_token = response.json()['access_token']
    print("Access token obtained successfully.")
else:
    print("Error obtaining access token.")
    exit()


Access token obtained successfully.


In [4]:
def get_trending_playlist_data(playlist_id,access_token):
    sp=spotipy.Spotify(auth=access_token)
    
    playlist_tracks=sp.playlist_tracks(playlist_id,fields='items(track(id,name,artists,album(id,name)))')
    
    music_data=[]
    
    for track_info in playlist_tracks['items']:
        track=track_info['track']
        track_name=track['name']
        album_name=track['album']['name']
        album_id=track['album']['id']
        track_id=track['id']
        artists=','.join([artist['name'] for artist in track['artists']])
        
        audio_features=sp.audio_features(track_id)[0] if track_id!='Not available' else None
        
        try:
            album_info=sp.album(album_id) if album_id!='Not available' else None
            release_date = album_info['release_date'] if album_info else None
        except:
            release_date = None

        try:
            track_info = sp.track(track_id) if track_id != 'Not available' else None
            popularity = track_info['popularity'] if track_info else None
        except:
            popularity = None
            
        track_data = {
            'Track Name': track_name,
            'Artists': artists,
            'Album Name': album_name,
            'Album ID': album_id,
            'Track ID': track_id,
            'Popularity': popularity,
            'Release Date': release_date,
            'Duration (ms)': audio_features['duration_ms'] if audio_features else None,
            'Explicit': track_info.get('explicit', None),
            'External URLs': track_info.get('external_urls', {}).get('spotify', None),
            'Danceability': audio_features['danceability'] if audio_features else None,
            'Energy': audio_features['energy'] if audio_features else None,
            'Key': audio_features['key'] if audio_features else None,
            'Loudness': audio_features['loudness'] if audio_features else None,
            'Mode': audio_features['mode'] if audio_features else None,
            'Speechiness': audio_features['speechiness'] if audio_features else None,
            'Acousticness': audio_features['acousticness'] if audio_features else None,
            'Instrumentalness': audio_features['instrumentalness'] if audio_features else None,
            'Liveness': audio_features['liveness'] if audio_features else None,
            'Valence': audio_features['valence'] if audio_features else None,
            'Tempo': audio_features['tempo'] if audio_features else None,
        }

        music_data.append(track_data)
        df = pd.DataFrame(music_data)

    return df

In [5]:
playlist_id = secret_value_2

music_df = get_trending_playlist_data(playlist_id, access_token)

print(music_df)

                                    Track Name  \
0                                          VVV   
1   Nightcrawler (feat. Swae Lee & Chief Keef)   
2                                         Lyfe   
3                                    IN THE UK   
4                   FE!N (feat. Playboi Carti)   
..                                         ...   
95  Do Not Disturb (feat. Lil Yachty & Offset)   
96                                       Audi.   
97                                     Layaway   
98       FRANCHISE (feat. Young Thug & M.I.A.)   
99                                    HYSTERIA   

                                     Artists  \
0                      mikeeysmind,Sanikwave   
1           Travis Scott,Swae Lee,Chief Keef   
2                   alyx,prodbysky,teefaygoo   
3                                 NLE Choppa   
4                 Travis Scott,Playboi Carti   
..                                       ...   
95  Smokepurpp,Murda Beatz,Lil Yachty,Offset   
96             

In [6]:
print(music_df.isnull().sum())

Track Name          0
Artists             0
Album Name          0
Album ID            0
Track ID            0
Popularity          0
Release Date        0
Duration (ms)       0
Explicit            0
External URLs       0
Danceability        0
Energy              0
Key                 0
Loudness            0
Mode                0
Speechiness         0
Acousticness        0
Instrumentalness    0
Liveness            0
Valence             0
Tempo               0
dtype: int64


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity

data = music_df

In [8]:
def calculate_weighted_popularity(release_date):
    release_date = datetime.strptime(release_date, '%Y-%m-%d')

    time_span = datetime.now() - release_date

    weight = 1 / (time_span.days + 1)
    return weight

In [9]:
scaler = MinMaxScaler()
music_features = music_df[['Danceability', 'Energy', 'Key', 
                           'Loudness', 'Mode', 'Speechiness', 'Acousticness',
                           'Instrumentalness', 'Liveness', 'Valence', 'Tempo']].values
music_features_scaled = scaler.fit_transform(music_features)

In [10]:
# def content_based_recommendations(input_song_name,num_recommendations=5):
#     if input_song_name not in music_df['Track Name'].values:
#         print(f"'{input_song_name}' not found in the dataset. Please enter a valid song name.")
#         return

#     input_song_index=music_df[music_df['Track Name']==input_song_name].index[0]
    
#     similarity_scores=cosine_similarity([music_features_scaled[input_song_index]],music_features_scaled)
    
#     similar_song_indices=similarity_scores.argsort()
    
#     content_based_recommendations=music_df.iloc[similar_song_indices][['Track Name', 'Artists', 'Album Name', 'Release Date', 'Popularity']]

#     return content_based_recommendations

def content_based_recommendations(input_song_name, num_recommendations=5):
    if input_song_name not in music_df['Track Name'].values:
        print(f"'{input_song_name}' not found in the dataset. Please enter a valid song name.")
        return

    # Get the index of the input song in the music DataFrame
    input_song_index = music_df[music_df['Track Name'] == input_song_name].index[0]

    # Calculate the similarity scores based on music features (cosine similarity)
    similarity_scores = cosine_similarity([music_features_scaled[input_song_index]], music_features_scaled)

    # Get the indices of the most similar songs
    similar_song_indices = similarity_scores.argsort()[0][::-1][1:num_recommendations + 1]

    # Get the names of the most similar songs based on content-based filtering
    content_based_recommendations = music_df.iloc[similar_song_indices][['Track Name', 'Artists', 'Album Name', 'Release Date', 'Popularity']]

    return content_based_recommendations

In [11]:
def hybrid_recommendations(input_song_name, num_recommendations, alpha=0.5):
    if input_song_name not in music_df['Track Name'].values:
        print(f"'{input_song_name}' not found in the dataset. Please enter a valid song name.")
        return

    content_based_rec = content_based_recommendations(input_song_name, num_recommendations)

    popularity_score = music_df.loc[music_df['Track Name'] == input_song_name, 'Popularity'].values[0]

    weighted_popularity_score = popularity_score * calculate_weighted_popularity(music_df.loc[music_df['Track Name'] == input_song_name, 'Release Date'].values[0])

    hybrid_recommendations = pd.concat([
        content_based_rec,
        pd.DataFrame({
            'Track Name': input_song_name,
            'Artists': music_df.loc[music_df['Track Name'] == input_song_name, 'Artists'].values[0],
            'Album Name': music_df.loc[music_df['Track Name'] == input_song_name, 'Album Name'].values[0],
            'Release Date': music_df.loc[music_df['Track Name'] == input_song_name, 'Release Date'].values[0],
            'Popularity': weighted_popularity_score
        }, index=[0])
      ], ignore_index=True)


    # Sort the hybrid recommendations based on weighted popularity score
    hybrid_recommendations = hybrid_recommendations.sort_values(by='Popularity', ascending=False)

    # Remove the input song from the recommendations
    hybrid_recommendations = hybrid_recommendations[hybrid_recommendations['Track Name'] != input_song_name]


    return hybrid_recommendations

In [12]:
input_song_name = "TIME"
recommendations = hybrid_recommendations(input_song_name, num_recommendations=15,alpha=0.5)
print(f"Hybrid recommended songs for '{input_song_name}':")
print(recommendations)

Hybrid recommended songs for 'TIME':
                     Track Name                             Artists  \
7                   Look At Me!                        XXXTENTACION   
5                      Mo Bamba                           Sheck Wes   
8                           VVV               mikeeysmind,Sanikwave   
9                  Wat U Want 2               mikeeysmind,prodbysky   
1                  Baby again..      Fred again..,Skrillex,Four Tet   
3   SICKO MODE - Skrillex Remix               Travis Scott,Skrillex   
13                   DISTORTION                        LXST CXNTURY   
14                        Birdz                     Wuki,Smokepurpp   
11                         Lyfe            alyx,prodbysky,teefaygoo   
0           Government Official                              Future   
2                    Fire on Me                              Inteus   
10                      Layaway  Alexander Lewis,Chief Keef,T-Shyne   
12                   Sammy Sosa         